In [1]:
device = "cuda"

### Preliminaries

In [2]:
import itertools
import random
import collections


import transformers
import torch
import tqdm.auto
from torch import Tensor

In [3]:
def sinusoidal_encode(
    x: Tensor,
    embedding_dim: int,
    min_value: int,
    max_value: int,
    use_l2_norm: bool = False,
    norm_const: float | None = None,
) -> Tensor:
    """
    Encodes a tensor of numbers into a sinusoidal representation, inspired by how absolute positional
    encoding works in transformers.

    The encoding is an evaluation of a sine and cosine function at different frequencies, where the
    frequency is determined by the embedding dimension and the allowed range of the input values.

    >>> sinusoidal_encode(
    ...     torch.tensor([-5, 2, 1, 0]),
    ...     embedding_dim=6,
    ...     min_value=-5,
    ...     max_value=5,
    ... )
    tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
            [ 0.6570,  0.7539, -0.1073, -0.9942,  0.9980,  0.0627],
            [-0.2794,  0.9602,  0.3491, -0.9371,  0.9616,  0.2746],
            [-0.9589,  0.2837,  0.7317, -0.6816,  0.8806,  0.4738]])
    """

    if embedding_dim % 2 != 0 and not use_l2_norm:
        raise ValueError("Embedding dimension must be even")

    if use_l2_norm:
        if embedding_dim % 2 == 0:
            reserved_dim = 2
        else:
            reserved_dim = 1
        embedding_dim -= reserved_dim
    else:
        reserved_dim = 0  # will not be used

    domain = max_value - min_value
    y_shape = x.shape + (embedding_dim,)
    y = torch.zeros(y_shape, device=x.device)
    even_indices = torch.arange(0, embedding_dim, 2)
    log_term = torch.log(torch.tensor(domain)) / embedding_dim
    div_term = torch.exp(even_indices * -log_term)
    x = x - min_value
    values = x.unsqueeze(-1).float() * div_term
    y[..., 0::2] = torch.sin(values)
    y[..., 1::2] = torch.cos(values)

    if use_l2_norm:
        y = torch.cat([y, torch.ones_like(y[..., :reserved_dim])], dim=-1)
        y /= y.norm(dim=-1, keepdim=True, p=2)

    if norm_const is not None:
        y *= norm_const

    return y

def binary_encode(
    x: Tensor,
    embedding_dim: int,
    min_value: int | float,
    max_value: int | float,
    use_l2_norm: bool = False,
    norm_const: float | None = None,
) -> Tensor:
    y = torch.zeros(x.shape + (embedding_dim,), device=x.device)
    reserve_dim = 0 if not use_l2_norm else 1
    x = x - min_value
    maximum = x.max()
    for i in range(embedding_dim - reserve_dim):
        coeff = 2**i
        if maximum < coeff:
            break
        y[..., -i - 1] = torch.floor(x / coeff) % 2
        x = x - coeff * y[..., -i - 1]
    if use_l2_norm:
        y = torch.cat([y, torch.ones_like(y[..., :reserve_dim])], dim=-1)
        y /= y.norm(dim=-1, keepdim=True, p=2)
    if norm_const is not None:
        y *= norm_const
    return y

### Prepare model and data

In [4]:
model_ckpt = "meta-llama/Llama-3.2-1B"
model = transformers.AutoModel.from_pretrained(model_ckpt, token="XXX").eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(model_ckpt)
tokenizer.add_special_tokens({'pad_token': '<|end_of_text|>'})
model = model.half().to(device).eval()

In [5]:
all_values = torch.arange(0, 1000)
mask = torch.rand(len(all_values), generator=torch.Generator().manual_seed(0))
train_mask = mask < 0.9
valid_mask = ~train_mask & (mask < 0.95)
test_mask = ~train_mask & ~valid_mask

train_values = all_values[train_mask]
valid_values = all_values[valid_mask]
test_values = all_values[test_mask]

In [6]:
all_inputs = all_values.tolist()
all_inputs_val = [(x1, x2) for x1, x2 in itertools.product(all_values.tolist(), repeat=2) if x1 + x2 < 1000]
train_values_set = set(train_values.tolist())
valid_values_set = set(valid_values.tolist())
test_values_set = set(test_values.tolist())
        
train_inputs = [x for x in all_inputs if x in train_values_set]
valid_inputs = [(x1, x2) for x1, x2 in all_inputs_val if x2 in valid_values_set]
test_inputs = [(x1, x2) for x1, x2 in all_inputs_val if x2 in test_values_set]

# sanity check
assert set(train_inputs) & set(valid_inputs) == set()
assert set(train_inputs) & set(test_inputs) == set()
assert set(valid_inputs) & set(test_inputs) == set()

random.seed(0)
random.shuffle(train_inputs)
random.shuffle(valid_inputs)
random.shuffle(test_inputs)
valid_size = 4096
train_size = 100_000
train_inputs = train_inputs[:train_size]
valid_inputs = valid_inputs[:valid_size]

In [7]:
len(train_inputs)

888

### Constructing altered natural texts -- with all numbers from pre-defined ranges

In [8]:
# cell loading the input texts
import json
from glob import glob
from tqdm import tqdm

import torch
import datasets
from git import Repo
import os

import itertools


HOME_PATH = "./"

def load_data(genre="food-1", downsample_to=0):
    """
    genre: input , genre of dataset you want to load
    data :  output,

    """
    if genre ==  'food-1':
        directory_path = "./FoodRecipe-ImageCaptioning/"
        if os.path.exists(directory_path) and os.path.isdir(directory_path):
            1;
        else:
            Repo.clone_from("https://github.com/samsatp/FoodRecipe-ImageCaptioning.git/", "./FoodRecipe-ImageCaptioning/")

        with open(HOME_PATH + directory_path + "data/data_strings_local.json", "r") as fp:
            recipes = json.load(fp)
            #print(recipes)
            concated_data = [' '.join(d) for d in recipes.values()]
            data = concated_data
            print(len(data))

    elif genre == 'food-2':
        reciepe_data2 = datasets.load_dataset("m3hrdadfi/recipe_nlg_lite",trust_remote_code=True) #steps o ingredients
        #train 6118 test 1000
        # ['uid', 'name', 'description', 'link', 'ner', 'ingredients', 'steps']
        data  = reciepe_data2['train']['steps']

    elif genre == 'arthmetic-1':

        metamathqa = datasets.load_dataset("meta-math/MetaMathQA") #original_question
        data = metamathqa['train']['original_question']

    elif genre == 'arthmetic-2':

        drop = datasets.load_dataset("ucinlp/drop") #passage
        data = drop['train']['passage']#['section_id', 'query_id', 'passage', 'question', 'answers_spans']

    elif genre == 'arthmetic-3':
        aquarat = datasets.load_dataset("deepmind/aqua_rat") #['question', 'options', 'rationale', 'correct'] go question or rationale
        data = aquarat['train']['question']

    elif genre == 'technical-1':
        icdatta = datasets.load_dataset("atta00/icd10-codes") #['chapter', 'section', 'category', 'category_code', 'code', 'description']
        data = [f"description: {d} | code: {c}" for d,c in zip(icdatta['train']['description'], icdatta['train']['code'] )] # go for description + code

    elif genre == 'technical-2':
        icdcm = datasets.load_dataset("Gokul-waterlabs/ICD-10-CM")#input+output
        data = [f"Description: {d} | code: {c}" for d,c in zip(icdcm['train']['input'], icdcm['train']['output'] )]

    elif genre == 'datetime-1':

        directory_path = "./TimeLineExtractionDecisionLettersCASE/"
        if os.path.exists(directory_path) and os.path.isdir(directory_path):
            1;
        else:
            Repo.clone_from("https://github.com/irlabamsterdam/TimeLineExtractionDecisionLettersCASE.git", directory_path)

        data = []
        for file in tqdm(glob(HOME_PATH + directory_path + 'data/txt_files/train/*txt')):
            with open(file, 'r') as fp:
                data.append(fp.read())
    else:
        data="ERROR : Pick a genre from [food-1/2, arthmetic-1/2/3, techincal-1/2, datetime]"
        print(data)
    print("Number of samples in the data loaded:", len(data))
    if downsample_to and len(data) > downsample_to:
        print("Downsampling to %s" % downsample_to)
        data = data[:downsample_to]

    return data

texts = list(itertools.chain(*(load_data(k) for k in ['food-1', 'food-2', 'arthmetic-1', 'arthmetic-2', 'arthmetic-3', 'technical-1', 'technical-2', 'datetime-1'])))
print(len(texts))

719
Number of samples in the data loaded: 719


Repo card metadata block was not found. Setting CardData to empty.


Number of samples in the data loaded: 6118
Number of samples in the data loaded: 395000
Number of samples in the data loaded: 77400
Number of samples in the data loaded: 97467
Number of samples in the data loaded: 25719
Number of samples in the data loaded: 74044


100%|██████████| 50/50 [00:00<00:00, 3672.90it/s]

Number of samples in the data loaded: 50
676517


In [9]:
import re

def make_str_input(all_possible_operands: list[int]) -> str:
    selected_text = random.choice(texts)
    text_with_replaced_nums = re.sub(r"\d+", lambda _: str(random.choice(all_possible_operands)), selected_text)
    return text_with_replaced_nums

make_str_input(train_inputs), make_str_input(valid_inputs)

('Jared likes to draw monsters. He drew a monster family portrait. The mom had 263 eye and the dad had 958. They had 562 kids, each with 180 eyes. How many eyes did the whole family have?',
 'Compared with its metropolitan area, the city of Houstons population has a higher proportion of minorities. According to the (406, 498) United States Census, whites made up (669, 81)% of the city of Houstons population; (60, 170)% of the total population was non-Hispanic whites. Blacks or African Americans made up (227, 570)% of Houstons population, Native Americans in the United States made up (64, 383).(162, 276)% of the population,  Asians made up (255, 377)% ((145, 377).(267, 455)% Vietnamese Americans, (455, 498).(923, 73)% Chinese Americans, (728, 231).(9, 900)% Indian Americans, (704, 21).(445, 73)% Pakistani Americans, (4, 37).(560, 218)% Filipino Americans, (840, 31).(380, 455)% Korean Americans, (12, 707).(478, 338)% Japanese Americans) and Pacific Islanders made up (565, 345).(181, 74)%

In [10]:
def make_str_input_nums(operands: tuple[int, int] | list[int]) -> str:
    x1, x2 = operands
    return f"{x1} + {x2}"

make_str_input_nums((3, 500)), make_str_input_nums((3, 0))

('3 + 500', '3 + 0')

### Inference of model's hidden states

In [11]:
import tqdm

def get_hidden_states(model, str_inputs: list[str], batch_size: int) -> tuple[dict[int, Tensor], Tensor]:
    model.eval()
    num_input_ids = tokenizer(list(map(str, all_inputs)), add_special_tokens=False, return_tensors="pt").input_ids[:, 0]

    nums: list[str] = []
    hidden_states = collections.defaultdict(list)
    with torch.no_grad():
        num_batches = (len(str_inputs) + batch_size - 1) // batch_size
        for batch_str in tqdm.auto.tqdm(itertools.batched(str_inputs, n=batch_size), total=num_batches):
            batch_inputs = tokenizer(batch_str, return_tensors="pt", padding=True, truncation=True)
            num_pos = torch.isin(batch_inputs.input_ids, num_input_ids)
            hidden_reprs = model(**batch_inputs.to(model.device), output_hidden_states=True).hidden_states
            for layer_idx, hidden_state in enumerate(hidden_reprs):
                hidden_states[layer_idx].extend(hidden_state[num_pos].detach().cpu())
            new_nums = tokenizer.batch_decode(batch_inputs.input_ids[num_pos])
            nums.extend(new_nums)

    return {k: torch.stack(v) for k, v in hidden_states.items()}, torch.tensor(list(map(int, nums)), device=device)

In [12]:
def get_hidden_states_raw_numbers(model, str_inputs: list[str], batch_size: int) -> collections.defaultdict[int, Tensor]:
    model.eval()
    hidden_states = collections.defaultdict(list)
    with torch.no_grad():
        num_batches = (len(str_inputs) + batch_size - 1) // batch_size
        for batch_str in tqdm.auto.tqdm(itertools.batched(str_inputs, n=batch_size), total=num_batches):
            batch_inputs = tokenizer(batch_str, return_tensors="pt")
            hidden_reprs = model(**batch_inputs.to(model.device), output_hidden_states=True).hidden_states
            for layer_idx, hidden_state in enumerate(hidden_reprs):
                hidden_states[layer_idx].extend(hidden_state[:, -1, :].detach().cpu())
    return {k: torch.stack(v) for k, v in hidden_states.items()}

In [13]:
batch_size = 8

train_input_texts = [make_str_input(train_inputs) for _ in range(10_000)]
train_hidden_states, train_labels = get_hidden_states(model, train_input_texts, batch_size)
assert train_hidden_states[0].shape[0] == len(train_labels)

valid_hidden_states = get_hidden_states_raw_numbers(model, [make_str_input_nums(val) for val in valid_inputs], batch_size)
test_hidden_states = get_hidden_states_raw_numbers(model, [make_str_input_nums(val) for val in test_inputs], batch_size)

# hidden_state, new_nums = get_hidden_states(model, train_input_texts, batch_size)


  0%|          | 0/1250 [00:00<?, ?it/s]

  0%|          | 0/512 [00:00<?, ?it/s]

  0%|          | 0/3032 [00:00<?, ?it/s]

### Probing

In [14]:
basis_embs_sin = sinusoidal_encode(
    torch.arange(1000),
    min_value=0,
    max_value=1000,
    embedding_dim=train_hidden_states[0].shape[-1],
)

basis_embs_bin = binary_encode(
    torch.arange(1000),
    min_value=0,
    max_value=1000,
    embedding_dim=10,
)

In [15]:
class ClassifierProbe(torch.nn.Module):
    def __init__(self, emb_dim: int, hidden_dim: int, basis: torch.Tensor, heldout_mask: torch.Tensor):
        super().__init__()
        self.emb_to_latent = torch.nn.Linear(emb_dim, hidden_dim, bias=True)
        self.basis_to_latent = torch.nn.Linear(basis.shape[-1], hidden_dim, bias=True)
        self.basis: torch.nn.Buffer
        self.heldout_mask: torch.nn.Buffer
        self.register_buffer("basis", basis)
        self.register_buffer("heldout_mask", heldout_mask)
    def forward(self, x: Tensor, holdout_eval_tokens: bool) -> Tensor:
        latent_x = self.emb_to_latent(x)
        # during training, model learns to choose among only training tokens
        # but during eval, model must choose among all tokens
        # this means that the model is never exposed to the eval tokens during training
        latent_choices = self.basis_to_latent(self.basis)
        logits = latent_x @ latent_choices.T
        if holdout_eval_tokens:
            logits[:, self.heldout_mask] = float("-inf")
        return logits

In [16]:
# train_labels = torch.tensor([x2 for x1, x2 in train_inputs])
valid_labels = torch.tensor([x2 for x1, x2 in valid_inputs]).to(device)
test_labels = torch.tensor([x2 for x1, x2 in test_inputs]).to(device)

test_accuracies = {"sin": {}, "bin": {}, "lin": {}, "log": {}}

for basis_name, basis_embs in {"sin": basis_embs_sin, "bin": basis_embs_bin}.items():
    for layer_idx in range(len(train_hidden_states)):

        torch.manual_seed(0)
        probe = ClassifierProbe(
            emb_dim=train_hidden_states[0].shape[-1],
            hidden_dim=100,
            basis=basis_embs,
            heldout_mask=test_mask,
        ).to(device)

        optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3)

        rng = torch.Generator().manual_seed(0)
        best_val_acc = -1
        best_ckpt = None
        for i in range(10000+1):
            probe.train()
            optimizer.zero_grad()
            minibatch_idcs = torch.randint(len(train_labels), size=(1024,), generator=rng)
            x = train_hidden_states[layer_idx][minibatch_idcs].float().to(device)
            y = train_labels[minibatch_idcs].to(device)
            logits = probe(x, holdout_eval_tokens=True)
            # add l1 regularization of all params to the loss
            loss = torch.nn.functional.cross_entropy(logits, y) + 0.001 * sum(p.abs().sum() for p in probe.parameters())
            loss.backward()
            optimizer.step()
            if i % 500 == 0:
                train_acc = (logits.argmax(dim=-1) == y).float().mean().item()
                probe.eval()
                with torch.no_grad():
                    valid_logits = probe(valid_hidden_states[layer_idx].float().to(device), holdout_eval_tokens=False)
                    valid_loss = torch.nn.functional.cross_entropy(valid_logits, valid_labels)
                    valid_accuracy = (valid_logits.argmax(dim=-1) == valid_labels).float().mean().item()
                    if valid_accuracy > best_val_acc:
                        best_val_acc = valid_accuracy
                        best_ckpt = probe.state_dict()
                print(f"{basis_name} {i=:>5} train loss: {loss.item():5.2f}  train acc: {train_acc:.2f}  val loss: {valid_loss.item():5.2f}  valid acc: {valid_accuracy:.2f}")
        probe.load_state_dict(best_ckpt)
        probe.eval()
        with torch.no_grad():
            test_logits = probe(test_hidden_states[layer_idx].float().to(device), holdout_eval_tokens=False)
            test_accuracy = (test_logits.argmax(dim=-1) == test_labels).float().mean().item()
        test_accuracies[basis_name][layer_idx] = test_accuracy
        print(f"->  {basis_name}  layer idx: {layer_idx:<3}, best valid accuracy: {best_val_acc:.2f}, test accuracy: {test_accuracy:.2f}")


sin i=    0 train loss: 11.39  train acc: 0.00  val loss:  6.87  valid acc: 0.00
sin i=  500 train loss:  2.40  train acc: 0.98  val loss:  1.49  valid acc: 0.77
sin i= 1000 train loss:  1.95  train acc: 1.00  val loss:  1.03  valid acc: 0.94
sin i= 1500 train loss:  1.74  train acc: 1.00  val loss:  0.86  valid acc: 0.90
sin i= 2000 train loss:  1.62  train acc: 1.00  val loss:  0.81  valid acc: 0.90
sin i= 2500 train loss:  1.54  train acc: 1.00  val loss:  0.79  valid acc: 0.85
sin i= 3000 train loss:  1.46  train acc: 1.00  val loss:  0.78  valid acc: 0.85
sin i= 3500 train loss:  1.39  train acc: 1.00  val loss:  0.76  valid acc: 0.84
sin i= 4000 train loss:  1.35  train acc: 1.00  val loss:  0.74  valid acc: 0.82
sin i= 4500 train loss:  1.30  train acc: 1.00  val loss:  0.69  valid acc: 0.86
sin i= 5000 train loss:  1.27  train acc: 1.00  val loss:  0.67  valid acc: 0.88
sin i= 5500 train loss:  1.23  train acc: 1.00  val loss:  0.65  valid acc: 0.88
sin i= 6000 train loss:  1.2

In [17]:
valid_hidden_states[0][layer_idx].shape, len(valid_labels)

(torch.Size([2048]), 4096)

In [18]:
def solve_linear_layer(x: Tensor, y: Tensor) -> torch.nn.Linear:
    if y.ndim == 1:
        y = y.unsqueeze(-1)
    if not y.is_floating_point():
        y = y.float()
   
    lin = torch.nn.Linear(x.shape[-1], y.shape[-1], device=x.device)
    x_aug = torch.cat([x, torch.ones(len(x), 1, device=x.device)], dim=1)
    coeffs = torch.linalg.lstsq(x_aug, y).solution
    w, b = coeffs[:-1], coeffs[-1]
    with torch.no_grad():
        lin.weight[:] = w.T
        lin.bias[:] = b
    return lin

In [19]:
for layer_idx in range(len(train_hidden_states)):
    lin_probe = solve_linear_layer(
        train_hidden_states[layer_idx].float().to(device),
        train_labels.to(device),
    )
    log_probe = solve_linear_layer(
        train_hidden_states[layer_idx].float().to(device),
        train_labels.log1p().to(device),
    )
    lin_test_pred = lin_probe(test_hidden_states[layer_idx].float().to(device)).flatten().round().int()
    lin_test_accuracy = (lin_test_pred == test_labels).float().mean().item()
    
    log_test_pred = log_probe(test_hidden_states[layer_idx].float().to(device)).flatten().exp().add(1).round().int()
    log_test_accuracy = (log_test_pred == test_labels).float().mean().item()
    
    test_accuracies["lin"][layer_idx] = lin_test_accuracy
    test_accuracies["log"][layer_idx] = log_test_accuracy

    print(f"layer idx: {layer_idx:<3}, linear probe acc: {lin_test_accuracy:.2f}, log probe acc: {log_test_accuracy:.2f}")

layer idx: 0  , linear probe acc: 0.00, log probe acc: 0.00
layer idx: 1  , linear probe acc: 0.02, log probe acc: 0.04
layer idx: 2  , linear probe acc: 0.02, log probe acc: 0.04
layer idx: 3  , linear probe acc: 0.01, log probe acc: 0.03
layer idx: 4  , linear probe acc: 0.01, log probe acc: 0.02
layer idx: 5  , linear probe acc: 0.01, log probe acc: 0.02
layer idx: 6  , linear probe acc: 0.01, log probe acc: 0.02
layer idx: 7  , linear probe acc: 0.01, log probe acc: 0.02
layer idx: 8  , linear probe acc: 0.01, log probe acc: 0.01
layer idx: 9  , linear probe acc: 0.01, log probe acc: 0.01
layer idx: 10 , linear probe acc: 0.01, log probe acc: 0.01
layer idx: 11 , linear probe acc: 0.01, log probe acc: 0.01
layer idx: 12 , linear probe acc: 0.01, log probe acc: 0.02
layer idx: 13 , linear probe acc: 0.01, log probe acc: 0.01
layer idx: 14 , linear probe acc: 0.00, log probe acc: 0.01
layer idx: 15 , linear probe acc: 0.01, log probe acc: 0.01
layer idx: 16 , linear probe acc: 0.01, 

In [20]:
for name, accs in test_accuracies.items():
    print(f"{name} accs: | " + " | ".join([f"{x:.0%}" for layer, x in sorted(accs.items())]) + " |")

sin accs: | 85% | 93% | 100% | 99% | 99% | 100% | 100% | 100% | 97% | 97% | 100% | 100% | 100% | 100% | 98% | 97% | 45% |
bin accs: | 33% | 18% | 15% | 18% | 8% | 3% | 2% | 1% | 2% | 6% | 9% | 6% | 10% | 4% | 9% | 5% | 1% |
lin accs: | 0% | 2% | 2% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 1% | 0% | 1% | 1% |
log accs: | 0% | 4% | 4% | 3% | 2% | 2% | 2% | 2% | 1% | 1% | 1% | 1% | 2% | 1% | 1% | 1% | 1% |
